In [4]:
import numpy as np
from pathlib import Path
from polar.tensor import tensor
from polar import graph
from polar.utils import hpf, detectar_onsets, alinear

RuntimeError: CPU dispatcher tracer already initlized

## Cargar o Crear el tensor

In [ ]:
medicion = "forte"
POLAR_DATA_PATH = Path(f"data/tensores/{medicion}.npy")
SR = 44100
if POLAR_DATA_PATH.exists():
    polar_data = np.load(POLAR_DATA_PATH, mmap_mode='r')
    print("Tensor cargado desde disco (mmap).")
    print(f"Shape del tensor cargado: {polar_data.shape}")
    print("tensor[azimuth, mic, sample]:")
else:
    polar_data, sr, azimuth, mics = tensor(f"data/audio/array/{medicion}")
    np.save(POLAR_DATA_PATH, polar_data)
    print("Tensor creado y guardado.")

## Graficar una toma o mic

In [ ]:
mics    = ['ref'] + list(range(1, 20))   # 20 elementos
angulos = list(range(0, 190, 10))        # 19 elementos

In [ ]:
# Todas las tomas del mic 10
graph.takes(polar_data[:, 10, :], modo='angulos', angulos=angulos, titulo="Todas las tomas para Mic 10", sr=SR)


In [ ]:

# Todos los mics de la toma 0°
graph.takes(polar_data[0, :, :], modo='mics', mics=mics, titulo="Todos los mics en toma 0°", sr=SR)

## Filtado de señal de mic referencia a 1m

In [ ]:
mic_ref_filt = hpf(polar_data[:, 0, :], sr=SR) 
mic_10_filt = hpf(polar_data[:, 10, :], sr=SR)

In [ ]:
graph.takes(mic_ref_filt, titulo="Tomas del mic_ref filtrado", modo='mics', mics=['ref'], sr=SR)

In [ ]:
graph.takes(mic_10_filt, titulo="Señal filtrada (mic 10)", modo='mics', mics=['10'], sr=SR)

In [ ]:



# 1. Detectar onsets del mic_10
i_mic10 = mics.index(10)
onsets_mic10 = detectar_onsets(polar_data[:, i_mic10, :], sr=SR, etiquetas=angulos)

# 2. Detectar onsets del mic_ref
i_ref = mics.index('ref')
onsets_ref = detectar_onsets(polar_data[:, i_ref, :], sr=SR, etiquetas=angulos)

# 3. Alinear todo el tensor
polar_alineado = alinear(polar_data, onsets_mic10, onsets_ref=onsets_ref, i_ref=i_ref, sr=SR, gap_ms=50)

In [ ]:

# Todos los mics de la toma 0°
graph.takes(polar_alineado[0, :, :], modo='mics', mics=mics, titulo="Todos los mics en toma 0°", sr=SR)